# BirdCLEF 2026 — Multi-Resolution Ensemble (Pipeline 03)

Trains **three independent EfficientNet-B0 models** on different temporal window sizes:

| Model | Window | Captures |
|---|---|---|
| M1 | 1 s | transient chirps, insects |
| M2 | 5 s | standard bird calls |
| M3 | 30 s | long phrases, habitat context, frogs |

Final predictions = **weighted mean** of the three model probability outputs.

> EDA finding: different taxa operate at different temporal scales. A single window size loses information.

## 1. Setup & Imports

In [ ]:
import os
import gc
import sys
import math
import time
import glob
import random
import ast
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, ConcatDataset
import torchaudio.transforms as T
import timm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

## 2. Configuration & Paths

In [ ]:
class Config:
    ROOT_DIR       = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV      = os.path.join(ROOT_DIR, 'train.csv')
    TRAIN_AUDIO_DIR = os.path.join(ROOT_DIR, 'train_audio')
    SOUNDSCAPE_CSV = os.path.join(ROOT_DIR, 'train_soundscapes_labels.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    MODEL_DIR = Path('/kaggle/input/models/timm/tf-efficientnet/pytorch/tf-efficientnet-b0/1/tf_efficientnet_b0_aa-827b6e33.pth')

    SR         = 32000
    WINDOW_SIZES = [1, 5, 30]   # M1, M2, M3
    ENSEMBLE_WEIGHTS = [0.15, 0.55, 0.30]

    N_MELS     = 128
    N_FFT      = 2048
    HOP_LENGTH = 512
    FMIN       = 20
    FMAX       = 16000

    SEED         = 42
    BATCH_SIZE   = 32
    EPOCHS       = 10
    LR           = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS  = 0

    MODEL_NAME   = 'tf_efficientnet_b0'
    NUM_CLASSES  = 0

CFG = Config()

print('Loading labels...')
train_df   = pd.read_csv(CFG.TRAIN_CSV)
ss_df      = pd.read_csv(CFG.SOUNDSCAPE_CSV)
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))

train_labels      = sorted(train_df['primary_label'].unique())
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
unique_labels     = submission_labels
label_to_id       = {label: i for i, label in enumerate(unique_labels)}
id_to_label       = {i: label for label, i in label_to_id.items()}

train_df['label_id'] = train_df['primary_label'].map(label_to_id)
CFG.NUM_CLASSES      = len(unique_labels)

print(f'Clip labels: {len(train_labels)} | Submission classes: {CFG.NUM_CLASSES}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 3. Utilities

In [ ]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG.SEED)


## 4. Multi-Resolution Datasets

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, audio_dir, window_seconds, is_train=True):
        self.df             = df.reset_index(drop=True)
        self.audio_dir      = audio_dir
        self.is_train       = is_train
        self.window_samples = CFG.SR * window_seconds
        self.mel_transform  = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        try:
            info = sf.info(audio_path)
            total = info.frames
            if total > self.window_samples:
                start = random.randint(0, total - self.window_samples) if self.is_train else 0
                y, _ = sf.read(audio_path, start=start,
                               frames=self.window_samples, always_2d=True)
            else:
                y, _ = sf.read(audio_path, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)

        if self.is_train and random.random() < 0.5:
            y = y + 0.005 * np.random.randn(len(y))

        y_t  = torch.tensor(y, dtype=torch.float32)
        mel  = self.mel_transform(y_t)
        mel  = self.amplitude_to_db(mel)
        mel  = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img  = torch.stack([mel, mel, mel])

        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        target[row['label_id']] = 1.0
        if 'secondary_labels' in row and pd.notna(row['secondary_labels']):
            try:
                for sl in ast.literal_eval(row['secondary_labels']):
                    if sl in label_to_id:
                        target[label_to_id[sl]] = 1.0
            except: pass
        return img, target


In [ ]:
class SoundscapeDataset(Dataset):
    def __init__(self, df, audio_dir, window_seconds):
        self.df             = df.reset_index(drop=True)
        self.audio_dir      = audio_dir
        self.window_samples = CFG.SR * window_seconds
        self.mel_transform  = T.MelSpectrogram(
            sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
            n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
        )
        self.amplitude_to_db = T.AmplitudeToDB(top_db=80)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row        = self.df.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'])
        h, m, s    = map(int, row['start'].split(':'))
        start_s    = (h * 3600 + m * 60 + s) * CFG.SR
        try:
            y, _ = sf.read(audio_path, start=start_s,
                           stop=start_s + self.window_samples, always_2d=True)
            y = y.mean(axis=1)
            if len(y) < self.window_samples:
                y = np.pad(y, (0, self.window_samples - len(y)))
        except:
            y = np.zeros(self.window_samples)

        y_t  = torch.tensor(y, dtype=torch.float32)
        mel  = self.mel_transform(y_t)
        mel  = self.amplitude_to_db(mel)
        mel  = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
        img  = torch.stack([mel, mel, mel])

        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        for label in str(row['primary_label']).split(';'):
            if label in label_to_id:
                target[label_to_id[label]] = 1.0
        return img, target


## 5. Model Architecture & Augmentations

In [ ]:
def mixup_data(x, y, alpha=0.4):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

class SpecAugment(nn.Module):
    def __init__(self, freq_mask=16, time_mask=32, num_masks=2):
        super().__init__()
        self.freq_mask = freq_mask
        self.time_mask = time_mask
        self.num_masks = num_masks

    def forward(self, x):
        B, C, F, T = x.shape
        for _ in range(self.num_masks):
            f = random.randint(0, self.freq_mask)
            f0 = random.randint(0, max(1, F - f))
            x[:, :, f0:f0+f, :] = 0
            t = random.randint(0, self.time_mask)
            t0 = random.randint(0, max(1, T - t))
            x[:, :, :, t0:t0+t] = 0
        return x

class BirdModel(nn.Module):
    def __init__(self, model_name, num_classes, model_path=None, pretrained=False):
        super().__init__()
        if model_path is not None and Path(model_path).exists():
            self.backbone = timm.create_model(model_name, checkpoint_path=model_path, pretrained=pretrained, in_chans=3)
        else:
            self.backbone = timm.create_model(model_name, pretrained=pretrained, in_chans=3)
        
        if 'efficientnet' in model_name:
            in_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
        elif 'convnext' in model_name:
            in_features = self.backbone.head.fc.in_features
            self.backbone.head.fc = nn.Identity()
        else:
            in_features = self.backbone.get_classifier().in_features
            self.backbone.reset_classifier(0)
            
        self.head = nn.Linear(in_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        out = self.head(features)
        return out


## 6. Training & Validation Loops

In [ ]:
def get_optimizer(model):
    return optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)

def get_scheduler(optimizer):
    return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.EPOCHS, eta_min=1e-6)

def get_criterion():
    return nn.BCEWithLogitsLoss()

spec_augment = SpecAugment(freq_mask=16, time_mask=32, num_masks=2)

def train_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    spec_augment.train()
    epoch_loss = 0.0
    for images, targets in tqdm(loader, desc='Train', leave=False):
        images, targets = images.to(device), targets.to(device)
        images = spec_augment(images)
        images, targets_a, targets_b, lam = mixup_data(images, targets)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = mixup_criterion(criterion, outputs, targets_a, targets_b, lam)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    return epoch_loss / len(loader)

def valid_epoch(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0.0
    preds, true_targets = [], []
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Valid', leave=False):
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            epoch_loss += loss.item()
            preds.append(torch.sigmoid(outputs).cpu().numpy())
            true_targets.append(targets.cpu().numpy())
    preds = np.concatenate(preds)
    true_targets = np.concatenate(true_targets)
    auc_scores = []
    for i in range(CFG.NUM_CLASSES):
        if len(np.unique(true_targets[:, i])) > 1:
            auc_scores.append(roc_auc_score(true_targets[:, i], preds[:, i]))
    final_auc = np.mean(auc_scores) if auc_scores else 0.0
    return epoch_loss / len(loader), final_auc


## 7. Main Execution (Training 3 Models)

In [ ]:
from sklearn.model_selection import train_test_split

# Split Data (Zero Leakage)
df = train_df.copy()
counts = df['label_id'].value_counts()
rare_birds = counts[counts < 2].index.tolist()
if rare_birds:
    df = pd.concat([df, df[df['label_id'].isin(rare_birds)]], ignore_index=True)

train_df_clips, valid_df_clips = train_test_split(df, test_size=0.2, stratify=df['label_id'], random_state=CFG.SEED)
train_ss_df, valid_ss_df = train_test_split(ss_df, test_size=0.2, random_state=CFG.SEED)

def get_sampler(df_subset):
    class_counts = df_subset['label_id'].value_counts().sort_index().values
    class_weights = 1.0 / (class_counts + 1e-6)
    weights = df_subset['label_id'].map(lambda x: class_weights[x] if x < len(class_weights) else 0).values
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

train_sampler = get_sampler(train_df_clips)

def run_training_for_resolution(window_sec):
    print(f"\n{'='*40}")
    print(f" TRAINING MODEL FOR WINDOW: {window_sec} SECONDS ")
    print(f"{'='*40}")
    
    train_clip_ds = BirdDataset(train_df_clips, CFG.TRAIN_AUDIO_DIR, window_sec, is_train=True)
    train_soundscape_ds = SoundscapeDataset(train_ss_df, CFG.SOUNDSCAPE_DIR, window_sec)
    train_ds = ConcatDataset([train_clip_ds, train_soundscape_ds])
    valid_ds_clips = BirdDataset(valid_df_clips, CFG.TRAIN_AUDIO_DIR, window_sec, is_train=False)
    valid_ds_ss = SoundscapeDataset(valid_ss_df, CFG.SOUNDSCAPE_DIR, window_sec)
    
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, sampler=train_sampler, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    valid_loader_clips = DataLoader(valid_ds_clips, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    valid_loader_ss = DataLoader(valid_ds_ss, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True)
    
    model = BirdModel(CFG.MODEL_NAME, CFG.NUM_CLASSES, CFG.MODEL_DIR).to(device)
    optimizer = get_optimizer(model)
    scheduler = get_scheduler(optimizer)
    criterion = get_criterion()
    scaler = torch.cuda.amp.GradScaler()
    
    best_score = 0
    for epoch in range(1, CFG.EPOCHS+1):
        start_time = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, scaler, device)
        _, val_auc_clips = valid_epoch(model, valid_loader_clips, criterion, device)
        _, val_auc_ss = valid_epoch(model, valid_loader_ss, criterion, device)
        scheduler.step()
        
        robust_score = val_auc_ss # We can just use validation SS AUC for simplicity here
        duration = time.time() - start_time
        print(f"Epoch {epoch} | Loss: {train_loss:.4f} | Time: {int(duration)}s")
        print(f"  Val Clips AUC: {val_auc_clips:.4f} | Val SS AUC: {val_auc_ss:.4f}")
        
        if robust_score > best_score:
            best_score = robust_score
            torch.save(model.state_dict(), f"best_model_{window_sec}s.pth")
            print(f"  !!! NEW BEST MODEL {window_sec}s SAVED | Val SS: {best_score:.4f} !!!")

for w in CFG.WINDOW_SIZES:
    run_training_for_resolution(w)
